# Analysis


DigiMic produces full consumer and resource trajectories, so analysis can be done at several levels: time-series summaries, final-state comparisons, perturbation experiments, and local stability calculations. This page outlines the main analysis ideas that fit the current model.


In [ ]:
import numpy as np
import param
from scipy.integrate import solve_ivp

np.random.seed(42)

N = 10
M = 5
λ = 0.1
N_modules = 2
s_ratio = 10.0

u = param.modular_uptake(N, M, N_modules, s_ratio)
lambda_alpha = np.full(M, λ)
m = np.full(N, 0.2)
rho = np.full(M, 0.5)
omega = np.full(M, 0.5)
l = param.generate_l_tensor(N, M, N_modules, s_ratio, λ)

def dCdt_Rdt(t, y):
    C = y[:N]
    R = y[N:]
    dCdt = np.zeros(N)
    dRdt = np.zeros(M)

    for i in range(N):
        dCdt[i] = sum(
            C[i] * R[alpha] * u[i, alpha] * (1 - lambda_alpha[alpha])
            for alpha in range(M)
        ) - C[i] * m[i]

    for alpha in range(M):
        dRdt[alpha] = rho[alpha] - R[alpha] * omega[alpha]
        dRdt[alpha] -= sum(C[i] * R[alpha] * u[i, alpha] for i in range(N))
        dRdt[alpha] += sum(
            sum(C[i] * R[beta] * u[i, beta] * l[i, beta, alpha] for beta in range(M))
            for i in range(N)
        )

    return np.concatenate([dCdt, dRdt])

C0 = np.full(N, 0.01)
R0 = np.full(M, 1.0)
Y0 = np.concatenate([C0, R0])

t_span = (0, 50)
t_eval = np.linspace(*t_span, 300)
sol = solve_ivp(dCdt_Rdt, t_span, Y0, t_eval=t_eval)
sol.success


## Time-series outputs

The `solve_ivp` result stores time points in `sol.t` and state trajectories in `sol.y`. Because consumers are stored before resources, the output can be split as:

```python
C_traj = sol.y[:N, :]
R_traj = sol.y[N:, :]
```

Common summaries include final abundance, time to equilibrium, maximum biomass, resource depletion, consumer persistence, and total community biomass.


In [ ]:
# Example summaries after running the usage notebook
C_traj = sol.y[:N, :]
R_traj = sol.y[N:, :]

total_biomass = C_traj.sum(axis=0)
resource_pool = R_traj.sum(axis=0)

print("Final total biomass:", round(total_biomass[-1], 4))
print("Final total resources:", round(resource_pool[-1], 4))
print("Peak total biomass:", round(total_biomass.max(), 4))


## Perturbation experiments

A direct way to study resilience is to perturb the simulated community and continue integration. Examples include reducing one consumer, adding a resource pulse, changing resource supply, or temporarily increasing mortality.

A simple pulse perturbation can be represented by copying the final state, modifying selected entries, and using that as a new initial condition.


In [ ]:
# Perturb consumer 1 and add a pulse to resource 1
Y_perturbed = sol.y[:, -1].copy()
Y_perturbed[0] *= 0.5
Y_perturbed[N + 0] += 0.5

post_span = (0, 25)
post_eval = np.linspace(*post_span, 150)
post = solve_ivp(dCdt_Rdt, post_span, Y_perturbed, t_eval=post_eval)

post.success


Perturbation outputs can be compared with the unperturbed equilibrium using recovery time, distance from the pre-perturbation state, or changes in consumer persistence.


## Local stability

Local stability analysis asks whether small deviations from a fixed point decay or grow. The usual workflow is:

1. Simulate until the system is close to equilibrium.
2. Compute or approximate the Jacobian matrix at that state.
3. Inspect eigenvalues of the Jacobian.

If every eigenvalue has a negative real part, the fixed point is locally stable. Eigenvalues with positive real parts indicate that small perturbations can grow.


In [ ]:
def numerical_jacobian(f, y_star, eps=1e-6):
    """Finite-difference Jacobian for a derivative function f(t, y)."""
    y_star = np.asarray(y_star, dtype=float)
    n = y_star.size
    J = np.zeros((n, n))
    for j in range(n):
        step = np.zeros(n)
        step[j] = eps
        J[:, j] = (f(0, y_star + step) - f(0, y_star - step)) / (2 * eps)
    return J

y_star = sol.y[:, -1]
J = numerical_jacobian(dCdt_Rdt, y_star)
eigvals = np.linalg.eigvals(J)

print("Maximum real eigenvalue:", np.max(eigvals.real))
print("Locally stable:", np.all(eigvals.real < 0))


## Reactivity and return rate

Local stability describes long-term behaviour near a fixed point, but stable systems can still show short-term amplification after perturbation. Reactivity measures whether an initial perturbation can grow transiently before eventually returning.

Two useful quantities are:

| Quantity | Interpretation |
|---|---|
| Maximum real eigenvalue | Long-term local stability |
| Numerical abscissa | Short-term reactivity, based on the symmetric part of the Jacobian |
| Return rate | Approximate speed of recovery for stable systems |

These metrics are especially useful when comparing communities with different uptake modularity or leakage structure.


In [ ]:
symmetric_part = (J + J.T) / 2
numerical_abscissa = np.max(np.linalg.eigvals(symmetric_part).real)
return_rate = -np.max(eigvals.real)

print("Reactive:", numerical_abscissa > 0)
print("Numerical abscissa:", round(numerical_abscissa, 4))
print("Return rate:", round(return_rate, 4))


## Scenario comparison

For research use, a single simulation is usually less informative than a controlled comparison across parameter scenarios and random seeds. A typical experiment might vary:

- leakage fraction `λ`;
- modularity strength `s_ratio`;
- number of modules `N_modules`;
- supply rates `rho`;
- consumer maintenance rates `m`.

For each scenario, record persistence, total biomass, resource pool size, local stability, reactivity, and recovery after perturbation. This gives a compact bridge between the mechanistic MiCRM parameters and emergent community-level behaviour.
